# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fkashaf19-afk/ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. Signal checks + my rule's reasoning

**My rule, in plain words.** A page goes in the refresh queue when it is **visible** (at least 500 impressions in 90 days) **and stale** (at least 180 days since its last update). Score = visibility percentile × staleness percentile.
- **Reason code (only one):** `stale_visible_page`
- **Action label:** `refresh` (everything not flagged is `monitor`)

**Signal 1 — staleness (behind the refresh flags).** If refresh flags are right, the declining share should rise as `days_since_last_update` rises.

**Signal 2 — volume (behind quick-win).** Volume says how big the prize is, not how likely a page is to decline. If confirmed, volume only belongs in the score as a size-of-prize multiplier.

In [2]:
import json, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

DATA_NAME = "data/raw/content_refresh_anonymized.csv"


def find_repo_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / DATA_NAME).exists():
            return p
    return None


ROOT = find_repo_root()
if ROOT is None:  # e.g. a fresh Colab session: fetch the repo (the CSV ships inside it)
    ROOT = Path("/content/ml-internship") if Path("/content").exists() else Path.cwd() / "ml-internship"
    if not ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/fkashaf19-afk/ml-internship", str(ROOT)],
            check=True,
        )

OUT_DIR = ROOT / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / DATA_NAME)
print("rows, cols:", df.shape, "  (starter file should be 30,000 x 44)")
print("trend/label-like columns present:", [c for c in df.columns if "trend" in c.lower() or "declin" in c.lower()])

REQUIRED = [
    "content_id", "client_id", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "word_count",
]
missing = [c for c in REQUIRED if c not in df.columns]
assert not missing, f"columns not found (check docs/data-dictionary.md for the exact names): {missing}"
assert df["content_id"].is_unique, "grain broken: content_id should be one row per item"

# AUDIT-ONLY outcome: used to CHECK signals and sanity-read the queue. Never enters the score.
if "is_declining_label" in df.columns:
    y = df["is_declining_label"].astype(str).str.lower().isin(["1", "1.0", "true"])
    Y_SOURCE = "is_declining_label"
elif "trend_direction" in df.columns:
    print("\ntrend_direction values:")
    print(df["trend_direction"].value_counts(dropna=False).to_string())
    y = df["trend_direction"].astype(str).str.strip().str.lower().eq("down")
    Y_SOURCE = "trend_direction == 'down'"
else:
    raise ValueError("No is_declining_label or trend_direction column. Check docs/data-dictionary.md.")

df["y_audit"] = y.astype(int)
assert 0 < df["y_audit"].sum() < len(df), "audit outcome is all-0 or all-1: check the trend_direction values printed above"
print(f"\naudit outcome built from: {Y_SOURCE}")
print(f"base rate of the audit outcome: {df['y_audit'].mean():.3f}")

rows, cols: (30000, 44)   (starter file should be 30,000 x 44)
trend/label-like columns present: ['trend_direction', 'trend_pct']

trend_direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

audit outcome built from: trend_direction == 'down'
base rate of the audit outcome: 0.542


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### My verdicts

- **Signal 1, staleness → declining:** VERDICT = `____` (n = ____). In words: ____
- **Signal 2, volume → declining:** VERDICT = `____` (n = ____). In words: ____
- **What this means for my rule:** ____

Observed, directional, same-window associations. Not proof that refreshing causes recovery.

In [4]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def bucket_table(d, col, outcome="y_audit", q=5):
    d = d[[col, outcome]].dropna()
    b = pd.qcut(d[col], q=q, duplicates="drop")
    g = d.groupby(b, observed=True)[outcome].agg(n="size", k="sum")
    g["rate"] = g["k"] / g["n"]
    ci = [wilson(k, n) for k, n in zip(g["k"], g["n"])]
    g["ci_low"] = [c[0] for c in ci]
    g["ci_high"] = [c[1] for c in ci]
    g = g.reset_index().rename(columns={col: "bucket"})
    g["bucket"] = g["bucket"].astype(str)
    return g


def suggest_verdict(tbl, expected=+1, min_buckets=3):
    """expected=+1: rate should rise with the signal; -1: fall."""
    if len(tbl) < min_buckets:
        return "MIXED"
    lo, hi = tbl.iloc[0], tbl.iloc[-1]
    if not (hi["ci_low"] > lo["ci_high"] or lo["ci_low"] > hi["ci_high"]):
        return "FALSE"  # top and bottom buckets are not distinguishable
    rho = tbl["rate"].reset_index(drop=True).corr(pd.Series(range(len(tbl))), method="spearman")
    moved = np.sign(hi["rate"] - lo["rate"]) * expected
    if moved > 0 and rho * expected >= 0.8:
        return "CONFIRMED"
    if moved < 0 and rho * expected <= -0.8:
        return "OPPOSITE"
    return "MIXED"


N_ALL = len(df)

# ---- Signal 1: staleness -> declining share
s1 = df.dropna(subset=["days_since_last_update"])
t1 = bucket_table(s1, "days_since_last_update")
v1 = suggest_verdict(t1, expected=+1)
print(f"SIGNAL 1 - staleness (days_since_last_update) vs declining share   n = {len(s1):,} of {N_ALL:,}")
display(t1.round(3))
print("suggested verdict:", v1, "\n")

# ---- Signal 2: volume -> declining share
s2 = df.dropna(subset=["impressions_90d"])
t2 = bucket_table(s2, "impressions_90d")
v2 = suggest_verdict(t2, expected=+1)
print(f"SIGNAL 2 - volume (impressions_90d) vs declining share   n = {len(s2):,} of {N_ALL:,}")
display(t2.round(3))
print("suggested verdict:", v2)

SIGNAL 1 - staleness (days_since_last_update) vs declining share   n = 30,000 of 30,000


,bucket,n,k,rate,ci_low,ci_high
0,"(0.999, 20.0]",15866,8550,0.539,0.531,0.547
1,"(20.0, 22.0]",3564,1402,0.393,0.377,0.410
2,"(22.0, 104.0]",10252,6136,0.599,0.589,0.608
3,"(104.0, 373.0]",318,174,0.547,0.492,0.601


suggested verdict: FALSE 

SIGNAL 2 - volume (impressions_90d) vs declining share   n = 30,000 of 30,000


,bucket,n,k,rate,ci_low,ci_high
0,"(0.999, 39.0]",6041,1964,0.325,0.313,0.337
1,"(39.0, 364.0]",5964,3597,0.603,0.591,0.615
2,"(364.0, 1375.0]",5997,3630,0.605,0.593,0.618
3,"(1375.0, 5167.6]",5998,3798,0.633,0.621,0.645
4,"(5167.6, 517715.0]",6000,3273,0.546,0.533,0.558


suggested verdict: MIXED


In [7]:
def why_line(r):
    return (f"{int(r['days_since_last_update']):,} days since last update, "
            f"{int(r['impressions_90d']):,} impressions in 90d "
            f"(volume pct {r['vis_pct'] * 100:.0f}, staleness pct {r['stale_pct'] * 100:.0f})")


def wrong_if(r):
    notes = []
    wc = r["word_count"]
    if pd.isna(wc) or wc <= 0:
        notes.append("word_count is missing, so 'stale' cannot be told apart from 'evergreen and fine'")
    if pd.isna(r["avg_position"]) or r["avg_position"] == 0:
        notes.append("avg_position is 0 = no data, so I cannot tell if the visibility is real ranking")
    elif r["avg_position"] <= 3:
        notes.append(f"it already ranks around position {r['avg_position']:.1f}, so there is little room to gain")
    if r["impressions_90d"] < 2 * MIN_IMPR:
        notes.append("impressions are close to the 500 gate, so a small change flips the flag")
    notes.append("the 'last update' date is a trivial edit, so the page is not truly stale")
    notes.append("the 90-day impressions are a one-off spike, not steady demand")
    return "; ".join(notes[:2])


top10 = q.head(10)
lines = []
for _, r in top10.iterrows():
    lines.append(
        f"{int(r['rank'])}. **{r['action_label']}** (`{r['reason_code']}`) — why: {why_line(r)}. "
        f"Wrong if: {wrong_if(r)}."
    )
display(Markdown("\n".join(lines)))

1. **refresh** (`stale_visible_page`) — why: 194 days since last update, 61,678 impressions in 90d (volume pct 99, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
2. **refresh** (`stale_visible_page`) — why: 194 days since last update, 59,472 impressions in 90d (volume pct 99, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
3. **refresh** (`stale_visible_page`) — why: 194 days since last update, 25,715 impressions in 90d (volume pct 96, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
4. **refresh** (`stale_visible_page`) — why: 193 days since last update, 13,299 impressions in 90d (volume pct 91, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
5. **refresh** (`stale_visible_page`) — why: 194 days since last update, 7,812 impressions in 90d (volume pct 85, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
6. **refresh** (`stale_visible_page`) — why: 193 days since last update, 7,558 impressions in 90d (volume pct 85, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
7. **refresh** (`stale_visible_page`) — why: 194 days since last update, 4,590 impressions in 90d (volume pct 78, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
8. **refresh** (`stale_visible_page`) — why: 194 days since last update, 4,556 impressions in 90d (volume pct 78, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
9. **refresh** (`stale_visible_page`) — why: 194 days since last update, 4,429 impressions in 90d (volume pct 78, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
10. **refresh** (`stale_visible_page`) — why: 193 days since last update, 1,697 impressions in 90d (volume pct 63, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.

In [8]:
def why_line(r):
    return (f"{int(r['days_since_last_update']):,} days since last update, "
            f"{int(r['impressions_90d']):,} impressions in 90d "
            f"(volume pct {r['vis_pct'] * 100:.0f}, staleness pct {r['stale_pct'] * 100:.0f})")


def wrong_if(r):
    notes = []
    wc = r["word_count"]
    if pd.isna(wc) or wc <= 0:
        notes.append("word_count is missing, so 'stale' cannot be told apart from 'evergreen and fine'")
    if pd.isna(r["avg_position"]) or r["avg_position"] == 0:
        notes.append("avg_position is 0 = no data, so I cannot tell if the visibility is real ranking")
    elif r["avg_position"] <= 3:
        notes.append(f"it already ranks around position {r['avg_position']:.1f}, so there is little room to gain")
    if r["impressions_90d"] < 2 * MIN_IMPR:
        notes.append("impressions are close to the 500 gate, so a small change flips the flag")
    notes.append("the 'last update' date is a trivial edit, so the page is not truly stale")
    notes.append("the 90-day impressions are a one-off spike, not steady demand")
    return "; ".join(notes[:2])


top10 = q.head(10)
lines = []
for _, r in top10.iterrows():
    lines.append(
        f"{int(r['rank'])}. **{r['action_label']}** (`{r['reason_code']}`) — why: {why_line(r)}. "
        f"Wrong if: {wrong_if(r)}."
    )
display(Markdown("\n".join(lines)))

1. **refresh** (`stale_visible_page`) — why: 194 days since last update, 61,678 impressions in 90d (volume pct 99, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
2. **refresh** (`stale_visible_page`) — why: 194 days since last update, 59,472 impressions in 90d (volume pct 99, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
3. **refresh** (`stale_visible_page`) — why: 194 days since last update, 25,715 impressions in 90d (volume pct 96, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
4. **refresh** (`stale_visible_page`) — why: 193 days since last update, 13,299 impressions in 90d (volume pct 91, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
5. **refresh** (`stale_visible_page`) — why: 194 days since last update, 7,812 impressions in 90d (volume pct 85, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
6. **refresh** (`stale_visible_page`) — why: 193 days since last update, 7,558 impressions in 90d (volume pct 85, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
7. **refresh** (`stale_visible_page`) — why: 194 days since last update, 4,590 impressions in 90d (volume pct 78, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
8. **refresh** (`stale_visible_page`) — why: 194 days since last update, 4,556 impressions in 90d (volume pct 78, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
9. **refresh** (`stale_visible_page`) — why: 194 days since last update, 4,429 impressions in 90d (volume pct 78, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.
10. **refresh** (`stale_visible_page`) — why: 193 days since last update, 1,697 impressions in 90d (volume pct 63, staleness pct 100). Wrong if: the 'last update' date is a trivial edit, so the page is not truly stale; the 90-day impressions are a one-off spike, not steady demand.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### My skeptic's note on the top ten

- I believe: ____
- I would drop first: ____ because ____
- One missing input that would help most: ____

In [5]:
STALE_DAYS = 180
MIN_IMPR = 500

SCORE_INPUTS = ["impressions_90d", "days_since_last_update"]
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label", "y_audit", "content_id", "client_id"}
assert not FORBIDDEN & set(SCORE_INPUTS), "a label-derived or ID column leaked into the score"

vis_pct = np.log1p(df["impressions_90d"]).rank(pct=True)
stale_pct = df["days_since_last_update"].rank(pct=True)
raw = (vis_pct * stale_pct)

gate = (df["days_since_last_update"] >= STALE_DAYS) & (df["impressions_90d"] >= MIN_IMPR)

q = df.copy()
q["vis_pct"] = vis_pct
q["stale_pct"] = stale_pct
q["_raw"] = raw.fillna(0.0)
q["score"] = np.where(gate, q["_raw"], 0.0)
q["reason_code"] = np.where(gate, "stale_visible_page", "no_flag")
q["action_label"] = np.where(gate, "refresh", "monitor")

q = q.sort_values(["score", "_raw", "content_id"], ascending=[False, False, True]).reset_index(drop=True)
q["rank"] = np.arange(1, len(q) + 1)

OUT_COLS = [
    "content_id", "client_id", "rank", "score", "reason_code", "action_label",
    "vis_pct", "stale_pct", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "word_count",
]
OUT_COLS += [c for c in ["content_type"] if c in q.columns]

CSV_PATH = OUT_DIR / "baseline_action_score.csv"
q[OUT_COLS].to_csv(CSV_PATH, index=False)  # no label column, no trend_* column

print(f"wrote {CSV_PATH.relative_to(ROOT)}  ({len(q):,} rows)")
print(q["action_label"].value_counts().to_string())
print(f"rows with unknown days_since_last_update (cannot be flagged): {df['days_since_last_update'].isna().sum():,}")
display(q[OUT_COLS].head(10))

wrote work/outputs/baseline_action_score.csv  (30,000 rows)
action_label
monitor    29983
refresh       17
rows with unknown days_since_last_update (cannot be flagged): 0


,content_id,client_id,rank,score,reason_code,action_label,vis_pct,stale_pct,impressions_90d,days_since_last_update,avg_position,ctr,word_count,content_type
0,content_cf56e2e2e282,client_7f2253d7e2,1,0.982686,stale_visible_page,refresh,0.986600,0.996033,61678,194,19.7,0.15,5125.0,keyword article
1,content_7368877ea310,client_7f2253d7e2,2,0.982288,stale_visible_page,refresh,0.986200,0.996033,59472,194,24.8,0.13,2591.0,keyword article
2,content_1bfaa38ff26c,client_7f2253d7e2,3,0.951809,stale_visible_page,refresh,0.955600,0.996033,25715,194,22.2,0.23,3861.0,keyword article
3,content_0a91db491d14,client_7f2253d7e2,4,0.904316,stale_visible_page,refresh,0.908100,0.995833,13299,193,10.5,0.49,3478.0,keyword article
4,content_5feee3994adb,client_7f2253d7e2,5,0.849450,stale_visible_page,refresh,0.852833,0.996033,7812,194,39.0,0.01,3590.0,keyword article
5,content_c2d929d83eaa,client_7f2253d7e2,6,0.845330,stale_visible_page,refresh,0.848867,0.995833,7558,193,17.9,0.20,4758.0,keyword article
6,content_b16bd7307b39,client_7f2253d7e2,7,0.780840,stale_visible_page,refresh,0.783950,0.996033,4590,194,31.0,0.00,4329.0,keyword article
7,content_fe16a55cd13d,client_7f2253d7e2,8,0.779745,stale_visible_page,refresh,0.782850,0.996033,4556,194,16.4,0.33,3388.0,keyword article
8,content_ecb6215e79fd,client_7f2253d7e2,9,0.775362,stale_visible_page,refresh,0.778450,0.996033,4429,194,25.3,0.38,4486.0,keyword article
9,content_928af3e22c80,client_7f2253d7e2,10,0.630147,stale_visible_page,refresh,0.632783,0.995833,1697,193,15.8,0.12,3118.0,keyword article


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### My weak picks

- Weakest pattern in my top picks: ____
- What I would change in the rule because of it: ____

In [9]:
top50 = q.head(50)
top10 = q.head(10)

print("Top-50 checks")
print(f"  word_count missing or 0 : {(top50['word_count'].isna() | (top50['word_count'] <= 0)).mean():.0%}")
print(f"  avg_position == 0 (none): {(top50['avg_position'] == 0).mean():.0%}")
print(f"  near the gate (<1000 imp): {(top50['impressions_90d'] < 2 * MIN_IMPR).mean():.0%}")
print(f"  distinct clients        : {top50['client_id'].nunique()}  (biggest client holds {top50['client_id'].value_counts(normalize=True).iloc[0]:.0%})")
if "content_type" in top50.columns:
    print("  content_type mix        :", top50["content_type"].value_counts(normalize=True).round(2).to_dict())

base = df["y_audit"].mean()
print(f"\nAudit outcome share: top-10 {top10['y_audit'].mean():.2f} | top-50 {top50['y_audit'].mean():.2f} | all rows {base:.2f}")
print("(full-data, same window, in-sample: a directional read only; the Week-5 model gets a proper held-out test)")

print("\nLeakage check")
print("  score inputs          :", SCORE_INPUTS)
print("  forbidden overlap     :", sorted(FORBIDDEN & set(SCORE_INPUTS)) or "none")
print("  label/trend in the CSV:", sorted({"is_declining_label", "y_audit", "trend_direction", "trend_pct"} & set(OUT_COLS)) or "none")
print("  windows               : all inputs are trailing-90-day columns; no future-window data used")

metrics = {
    "rule": {"gate": {"days_since_last_update_min": STALE_DAYS, "impressions_90d_min": MIN_IMPR},
             "score": "log1p(impressions) percentile x days_since_last_update percentile, gated",
             "reason_code": "stale_visible_page", "action_label": "refresh"},
    "signal_1_staleness": {"verdict_suggested": v1, "n": int(len(s1)), "table": t1.round(4).to_dict("records")},
    "signal_2_volume": {"verdict_suggested": v2, "n": int(len(s2)), "table": t2.round(4).to_dict("records")},
    "queue": {"rows": int(len(q)), "flagged_refresh": int((q["action_label"] == "refresh").sum()),
              "audit_outcome_top10": float(top10["y_audit"].mean()),
              "audit_outcome_top50": float(top50["y_audit"].mean()),
              "audit_outcome_all": float(base),
              "note": "in-sample directional read, not a held-out precision"},
    "score_inputs": SCORE_INPUTS,
}
with open(OUT_DIR / "w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nwrote", (OUT_DIR / "w04_baseline_metrics.json").relative_to(ROOT))

Top-50 checks
  word_count missing or 0 : 14%
  avg_position == 0 (none): 0%
  near the gate (<1000 imp): 10%
  distinct clients        : 11  (biggest client holds 46%)
  content_type mix        : {'keyword article': 1.0}

Audit outcome share: top-10 1.00 | top-50 0.76 | all rows 0.54
(full-data, same window, in-sample: a directional read only; the Week-5 model gets a proper held-out test)

Leakage check
  score inputs          : ['impressions_90d', 'days_since_last_update']
  forbidden overlap     : none
  label/trend in the CSV: none
  windows               : all inputs are trailing-90-day columns; no future-window data used

wrote work/outputs/w04_baseline_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.